In [ ]:
import arch


pip install arch

   ---------------------------------------- 0.0/937.9 kB ? eta -:--:--
   --------------------------------------- 937.9/937.9 kB 10.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   --------- ------------------------------ 2.4/9.6 MB 11.2 MB/s eta 0:00:01
   ------------------- -------------------- 4.7/9.6 MB 11.4 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.6 MB 11.6 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.6 MB 11.3 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 11.0 MB/s eta 0:00:00

   ------------- -------------------------- 1/3 [statsmodels]
   ------------- -------------------------- 1/3 [statsmodels]
   ------------- -------------------------- 1/3 [statsmodels]
   ------------- -------------------------- 1/3 [statsmodels]
   ------------- -------------------------- 1/3 [statsmodels]
   ------------- -------------------------- 1/3 [statsmodels]
   ------------- -----

In [ ]:
import yfinance as yf
import pandas as pd
import os

# carpeta destino
path = r"C:\Users\lucas\Downloads\GdP\Implementacion\Bases de datos"

# universos
byma = ["GGAL.BA","YPFD.BA","PAMP.BA","TXAR.BA","ALUA.BA","CEPU.BA","LOMA.BA",
        "BMA.BA","BBAR.BA","SUPV.BA","COME.BA","EDN.BA","TGNO4.BA","TGSU2.BA","VALO.BA"]

cedears = ["AAPL","MSFT","AMZN","GOOGL","META","NVDA","TSLA","BRK-B","JPM",
           "V","MA","KO","PEP","XOM","UNH"]

etfs = ["SPY","QQQ","IWM","EEM","EFA","GLD","SLV","TLT","IEF","HYG","VNQ"]

tickers = byma + cedears + etfs

# descarga completa
prices = yf.download(tickers, start="2015-01-01", auto_adjust=True)["Close"]


# metadata de activos
meta = pd.DataFrame({
    "ticker": tickers,
    "asset_type": ["BYMA"]*len(byma) + ["CEDEAR"]*len(cedears) + ["ETF"]*len(etfs)
})

# guardar archivos
prices.to_csv(os.path.join(path, "prices_all_assets.csv"))
meta.to_csv(os.path.join(path, "assets_metadata.csv"), index=False)

print("✔ Base completa guardada")


[*********************100%***********************]  41 of 41 completed
C:\Users\lucas\AppData\Local\Temp\ipykernel_27800\4133359937.py:23: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change().dropna()


✔ Base completa guardada


In [4]:
print("Fecha inicio dataset:", prices.dropna().index.min())
print("Fecha fin dataset:", prices.index.max())


Fecha inicio dataset: 2017-11-01 00:00:00
Fecha fin dataset: 2026-02-13 00:00:00


In [5]:
prices_clean = prices.dropna()

print("\nResumen dataset final")
print("---------------------")
print("Fecha inicio:", prices_clean.index.min().date())
print("Fecha fin:", prices_clean.index.max().date())
print("Cantidad días:", len(prices_clean))
print("Cantidad activos:", prices_clean.shape[1])



Resumen dataset final
---------------------
Fecha inicio: 2017-11-01
Fecha fin: 2026-02-13
Cantidad días: 1964
Cantidad activos: 41


In [ ]:
import pandas as pd
import os

path = r"C:\Users\lucas\Downloads\GdP\Implementacion\Bases de datos"

prices = pd.read_csv(os.path.join(path,"prices_all_assets.csv"), index_col=0, parse_dates=True)

prices_2018_2025 = prices.loc["2018-01-01":"2025-12-31"]
returns_2018_2025 = returns.loc["2018-01-01":"2025-12-31"]

prices_2018_2025.to_csv(os.path.join(path,"prices_2018_2025.csv"))

print("✔ Ventana 2018–2025 guardada")


✔ Ventana 2018–2025 guardada


In [ ]:
import pandas as pd
import numpy as np
import os

path = r"C:\Users\lucas\Downloads\GdP\Implementacion\Bases de datos"

# ===============================
# 1) PRECIOS EN ARS
# ===============================
prices_ars = pd.read_csv(os.path.join(path,"prices_2018_2025.csv"),
                         index_col=0, parse_dates=True)

prices_ars = prices_ars.ffill()

# ===============================
# 2) DOLAR CCL
# ===============================
ccl = pd.read_csv(os.path.join(path,"DOLAR CCL - Cotizaciones historicas.csv"))

# adaptá nombres si cambian
ccl.columns = [c.lower() for c in ccl.columns]
ccl.rename(columns={"fecha":"date","cierre":"ccl"}, inplace=True)

ccl["date"] = pd.to_datetime(ccl["date"])
ccl = ccl.set_index("date")

# alinear calendario con precios
ccl = ccl.reindex(prices_ars.index).ffill()

# ===============================
# 3) CONVERTIR TODO A USD
# ===============================
prices_usd = prices_ars.divide(ccl["ccl"], axis=0)

# ===============================
# 4) LOG RETURNS EN USD
# ===============================
returns_usd = np.log(prices_usd / prices_usd.shift(1)).dropna()

# ===============================
# 5) GUARDAR DATASET FINAL
# ===============================
prices_usd.to_csv(os.path.join(path,"prices_USD.csv"))
returns_usd.to_csv(os.path.join(path,"returns_USD.csv"))

print("✔ DATASET FINAL EN USD")
print("Rango:", prices_usd.index.min().date(), "→", prices_usd.index.max().date())
print("Activos:", prices_usd.shape[1])

✔ DATASET FINAL EN USD
Rango: 2018-01-01 → 2025-12-31
Activos: 41


In [10]:
import pandas as pd
import os

path = r"C:\Users\lucas\Downloads\GdP\Implementacion\Bases de datos"

byma = ["GGAL.BA","YPFD.BA","PAMP.BA","TXAR.BA","ALUA.BA","CEPU.BA","LOMA.BA",
        "BMA.BA","BBAR.BA","SUPV.BA","COME.BA","EDN.BA","TGNO4.BA","TGSU2.BA","VALO.BA"]

cedears = ["AAPL","MSFT","AMZN","GOOGL","META","NVDA","TSLA","BRK-B","JPM",
           "V","MA","KO","PEP","XOM","UNH"]

etfs = ["SPY","QQQ","IWM","EEM","EFA","GLD","SLV","TLT","IEF","HYG","VNQ"]

meta = pd.DataFrame({
    "ticker": byma + cedears + etfs,
    "asset_class": ["Argentina Equity"]*len(byma) +
                   ["US Equity"]*len(cedears) +
                   ["ETF"]*len(etfs)
})

meta.to_csv(os.path.join(path,"assets_metadata.csv"), index=False)

print("✔ assets_metadata.csv creado")


✔ assets_metadata.csv creado


In [12]:
import pandas as pd
import numpy as np
import os

path = r"C:\Users\lucas\Downloads\GdP\Implementacion\Bases de datos"

# 1) Precios mixtos (ARS para .BA, USD para US/ETF)
prices = pd.read_csv(os.path.join(path, "prices_2018_2025.csv"),
                     index_col=0, parse_dates=True).sort_index()
prices = prices.ffill()

# 2) CCL
ccl = pd.read_csv(os.path.join(path, "DOLAR CCL - Cotizaciones historicas.csv"))
ccl.columns = [c.lower() for c in ccl.columns]
ccl.rename(columns={"fecha": "date", "cierre": "ccl"}, inplace=True)
ccl["date"] = pd.to_datetime(ccl["date"])
ccl = ccl.set_index("date").sort_index()

# Alinear a calendario de precios
ccl = ccl.reindex(prices.index).ffill()

# 3) Convertir SOLO activos argentinos (.BA) a USD
prices_usd = prices.copy()
arg_assets = [col for col in prices.columns if col.endswith(".BA")]

prices_usd[arg_assets] = prices_usd[arg_assets].divide(ccl["ccl"], axis=0)

# 4) Log returns
returns_usd = np.log(prices_usd / prices_usd.shift(1)).dropna()

# 5) Guardar
prices_usd.to_csv(os.path.join(path, "prices.csv"))
returns_usd.to_csv(os.path.join(path, "returns.csv"))

print("✔ Dataset USD corregido")
print("Rango:", prices_usd.index.min().date(), "→", prices_usd.index.max().date())
print("Activos:", prices_usd.shape[1])
print("Convertidos a USD (ARS→USD):", len(arg_assets))
print("Dejados en USD (US/ETF):", prices_usd.shape[1] - len(arg_assets))



✔ Dataset USD corregido
Rango: 2018-01-01 → 2025-12-31
Activos: 41
Convertidos a USD (ARS→USD): 15
Dejados en USD (US/ETF): 26


In [14]:
pip install scipy


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

pred = pd.read_csv("outputs/predictions/predictions.csv")
meta = pd.read_csv("Data/assets_metadata.csv")

pred["ticker"] = pred["ticker"].astype(str).str.strip()
meta["ticker"] = meta["ticker"].astype(str).str.strip()

df = pred.merge(meta[["ticker", "block"]], on="ticker", how="left")

print(df.sort_values(["block", "mu_pred"]))
print("\nMean mu_pred by block:")
print(df.groupby("block")["mu_pred"].agg(["mean", "min", "max", "count"]))

      ticker       mse      rmse       mu_pred  var_pred  vol_pred  \
6    CEPU.BA  0.008417  0.091743 -6.264578e-05  0.000754  0.027451   
8     EDN.BA  0.010033  0.100166 -5.629563e-05  0.000599  0.024464   
33   TXAR.BA  0.006001  0.077464 -5.522454e-05  0.001145  0.033832   
11   GGAL.BA  0.008660  0.093059 -3.706270e-05  0.000812  0.028497   
29  TGNO4.BA  0.012368  0.111209 -3.372190e-05  0.000880  0.029672   
1    ALUA.BA  0.006646  0.081520 -3.206851e-05  0.000767  0.027694   
30  TGSU2.BA  0.008870  0.094181 -1.903707e-05  0.001019  0.031918   
39   YPFD.BA  0.005805  0.076190 -1.373479e-05  0.000701  0.026484   
28   SUPV.BA  0.011159  0.105637 -6.237610e-06  0.001097  0.033115   
24   PAMP.BA  0.005962  0.077212  7.138503e-06  0.000706  0.026578   
19   LOMA.BA  0.006760  0.082217  1.451128e-05  0.000543  0.023301   
7    COME.BA  0.017389  0.131867  2.100779e-05  0.001249  0.035344   
36   VALO.BA  0.007878  0.088759  2.165875e-05  0.000455  0.021337   
4     BMA.BA  0.0104

In [3]:
import pandas as pd

df = pd.read_csv("outputs/predictions/return_predictions_lstm.csv")

print(df.sort_values("rmse"))
print("\nMean RMSE:", df["rmse"].mean())


      ticker       mse      rmse       mae   mu_pred
14       HYG  0.000039  0.006207  0.004526  0.000035
15       IEF  0.000073  0.008520  0.006482 -0.000377
31       TLT  0.000334  0.018284  0.014091 -0.002882
37       VNQ  0.000458  0.021400  0.015524 -0.001191
18        KO  0.000462  0.021495  0.016578  0.001131
10       EFA  0.000472  0.021732  0.015875  0.001050
27       SPY  0.000482  0.021962  0.015873  0.001400
5      BRK-B  0.000537  0.023180  0.017915  0.003176
9        EEM  0.000565  0.023779  0.017886 -0.000757
12       GLD  0.000589  0.024269  0.018925  0.000922
25       PEP  0.000711  0.026658  0.020821  0.000641
20        MA  0.000732  0.027061  0.019861  0.004605
35         V  0.000791  0.028131  0.021665  0.004347
26       QQQ  0.000807  0.028413  0.021467  0.001893
16       IWM  0.000935  0.030586  0.022582  0.000438
22      MSFT  0.000953  0.030870  0.023559  0.003079
38       XOM  0.001033  0.032148  0.024823  0.006107
17       JPM  0.001149  0.033898  0.025283  0.

In [2]:
from infrastructure.storage.minio_client import upload_bytes

path = upload_bytes(
    bucket_name="optimization-runs",
    object_name="test/hello.txt",
    content=b"hola papu",
    content_type="text/plain",
)

print(path)

ModuleNotFoundError: No module named 'infrastructure'

In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
